# Contrast Stretching using Piecewise Linear Transformation

Slopes chosen are 0.5, 2, and 0.5

## 1. Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Prepare Image

In [ ]:
# Load image (Cameraman.tif)
try:
    I = io.imread('Cameraman.tif')
except FileNotFoundError:
    print("Cameraman.tif not found. Using a sample image instead.")
    # Create a sample image if file not found
    from skimage import data
    I = data.camera()

# Convert to grayscale if needed (if it has multiple channels)
if len(I.shape) > 2:
    I = np.dot(I[...,:3], [0.299, 0.587, 0.114])  # RGB to grayscale

# Convert to double and normalize
I = I.astype(np.float64)
I = (I * 2) / np.max(I)

print(f"Image shape: {I.shape}")
print(f"Image min: {I.min():.4f}, max: {I.max():.4f}")

## 3. Define Piecewise Linear Transformation Function

In [ ]:
# Define parameters for piecewise linear transformation
LT = 100  # Lower threshold value
UT = 150  # Upper threshold value

def piecewise_linear_transform(pixel_value, LT, UT):
    """
    Apply piecewise linear transformation with slopes 0.5, 2, and 0.5
    
    Parameters:
    - pixel_value: input pixel intensity
    - LT: lower threshold
    - UT: upper threshold
    
    Returns:
    - transformed pixel value
    """
    if pixel_value <= LT:
        return 0.5 * pixel_value
    elif pixel_value <= UT:
        return 2 * (pixel_value - LT) + (0.5 * LT)
    else:
        return 0.5 * (pixel_value - UT) + (0.5 * LT) + 2 * (UT - LT)

## 4. Apply Contrast Stretching

In [ ]:
# Apply piecewise linear transformation to the entire image
row, col = I.shape
I_str = np.zeros_like(I)

# Using vectorized operation for efficiency
for i in range(row):
    for j in range(col):
        I_str[i, j] = piecewise_linear_transform(I[i, j], LT, UT)

print(f"Contrast stretched image min: {I_str.min():.4f}, max: {I_str.max():.4f}")

## 5. Plot Transformation Function

In [ ]:
# Create transformation curve (similar to MATLAB code)
dd = np.zeros(256)

# Segment 1: 0 to LT (slope 0.5)
dd[0:LT] = 0.5 * np.arange(0, LT)

# Segment 2: LT to UT (slope 2)
dd[LT:UT] = 2 * (np.arange(LT, UT) - LT) + (0.5 * LT)

# Segment 3: UT to 256 (slope 0.5)
dd[UT:256] = 0.5 * (np.arange(UT, 256) - UT) + (0.5 * LT) + 2 * (UT - LT)

# Plot the transformation function
plt.figure(figsize=(10, 6))
plt.plot(dd, linewidth=2)
plt.grid(True)
plt.xlabel('Intensity in input image', fontsize=12)
plt.ylabel('Intensity in output image', fontsize=12)
plt.title('Contrast-Stretch Transformation Function', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Display Results

In [ ]:
# Display original and contrast-stretched images side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original image
axes[0].imshow(I, cmap='gray')
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

# Contrast-stretched image
axes[1].imshow(I_str, cmap='gray')
axes[1].set_title('Contrast Stretched Image', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()